# Playable Replays — offline decoded-packet QLoRA pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joyalzzy/playable-replays/blob/ml/player-ai.ipynb)

This notebook implements the safe, offline parts of the attached training guide using the [`maknee/league-of-legends-decoded-replay-packets`](https://huggingface.co/datasets/maknee/league-of-legends-decoded-replay-packets) dataset by default:

1. download one revision-pinned, checksum-verified decoded-packet batch;
2. compact and validate a bounded number of packet sequences;
3. form temporal windows with factual next-sampled-packet targets;
4. export reproducible instruction-tuning and optional preference JSONL;
5. optionally QLoRA-tune a 4-bit Llama 3.2 3B Instruct model with Unsloth; and
6. evaluate and export a LoRA adapter or optional GGUF artifact.

## Data and product boundary

This notebook does **not** parse `.rofl`/`.dem` files, download VODs, ingest runtime telemetry, or publish scenarios. It consumes an offline, decoded derivative whose publisher declares Apache-2.0; that declaration does not imply Riot Games endorsement. Downloaded batches and derived JSONL stay in the ignored local/Colab artifact directory and never enter the browser or Go runtime. Packet-native `x`/`z` values are not normalized simulator coordinates. Never upload credentials, personal data, proprietary raw replay data, or unlicensed assets.

The schema example below is only a structural smoke test and is excluded from training.

In [1]:
# Connectivity and runtime check — run this first on the assigned Colab kernel.
from __future__ import annotations

import importlib.util
import json
import os
import platform
import sys
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

IN_COLAB = importlib.util.find_spec("google.colab") is not None

def probe(url: str) -> dict[str, object]:
    request = Request(url, headers={"User-Agent": "playable-replays-colab-check/1.0"})
    try:
        with urlopen(request, timeout=12) as response:
            return {"reachable": True, "status": response.status, "finalUrl": response.url}
    except HTTPError as error:
        return {"reachable": True, "status": error.code, "finalUrl": error.url}
    except (URLError, TimeoutError, OSError) as error:
        return {"reachable": False, "error": type(error).__name__}

endpoints = {
    "colab": "https://colab.research.google.com/",
    "dataset": "https://huggingface.co/datasets/maknee/league-of-legends-decoded-replay-packets",
    "model": "https://huggingface.co/unsloth/Llama-3.2-3B-Instruct-bnb-4bit/resolve/main/config.json",
    "packages": "https://pypi.org/simple/unsloth/",
}
connectivity = {name: probe(url) for name, url in endpoints.items()}

try:
    import torch
    cuda_available = torch.cuda.is_available()
    accelerator = torch.cuda.get_device_name(0) if cuda_available else "CPU"
except ImportError:
    cuda_available = False
    accelerator = "torch-not-installed"

runtime_report = {
    "inColab": IN_COLAB,
    "python": platform.python_version(),
    "accelerator": accelerator,
    "connectivity": connectivity,
}
print(json.dumps(runtime_report, indent=2))
if not IN_COLAB:
    print("Not attached to a Colab runtime yet. In VS Code select: Kernel > Colab > Auto Connect.")
if IN_COLAB and not cuda_available:
    print("Connected to Colab without a GPU. Assign a GPU runtime before setting RUN_TRAINING=True.")

{
  "inColab": true,
  "python": "3.12.13",
  "accelerator": "CPU",
  "connectivity": {
    "colab": {
      "reachable": true,
      "status": 200,
      "finalUrl": "https://colab.research.google.com/"
    },
    "dataset": {
      "reachable": true,
      "status": 200,
      "finalUrl": "https://huggingface.co/datasets/maknee/league-of-legends-decoded-replay-packets"
    },
    "model": {
      "reachable": true,
      "status": 200,
      "finalUrl": "https://huggingface.co/api/resolve-cache/models/unsloth/Llama-3.2-3B-Instruct-bnb-4bit/bb1d317a108579fb40e646af8924a5e7ec5604b1/config.json?%2Funsloth%2FLlama-3.2-3B-Instruct-bnb-4bit%2Fresolve%2Fmain%2Fconfig.json=&etag=%222996b682f9232fc2a326fd4c786e922e206a375a%22"
    },
    "packages": {
      "reachable": true,
      "status": 200,
      "finalUrl": "https://pypi.org/simple/unsloth/"
    }
  }
}
Connected to Colab without a GPU. Assign a GPU runtime before setting RUN_TRAINING=True.


## 1. Configuration

The defaults download and verify one 86.6 MB compressed dataset batch, then stream only the configured number of games; they do not download a model. Use `schema_example` for a network-free smoke test or `local_records` for a separately licensed export. Set `RUN_TRAINING = True` only after inspecting the derived examples on a GPU runtime.

In [2]:
from pathlib import Path

BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  # @param {type:"string"}
DATA_SOURCE = "huggingface_replay_packets"  # @param ["huggingface_replay_packets", "local_records", "schema_example"]
DATA_PATH = ""  # @param {type:"string"}
HF_DATASET_REPO = "maknee/league-of-legends-decoded-replay-packets"
HF_DATASET_REVISION = "04f9c7350e9ffcc689b731875ad9baf3ff6eaa6d"
HF_REPLAY_FILE = "12_22/batch_001.jsonl.gz"
HF_REPLAY_SHA256 = "a0fdcb5fc12182fbdb8be0ce8c56c4490e5bd03abc7ccfeb82fe5c1346e3edbe"
MAX_REPLAY_GAMES = 8  # @param {type:"integer"}
MAX_PACKETS_PER_GAME = 2048  # @param {type:"integer"}
PACKET_SAMPLE_STRIDE = 4  # @param {type:"integer"}
OUTPUT_DIR = Path("/content/playable-replays-output" if IN_COLAB else "./.local-data/player-ai")
WINDOW_SIZE = 3  # @param {type:"integer"}
WINDOW_STRIDE = 1  # @param {type:"integer"}
EVAL_MATCH_FRACTION = 0.2  # @param {type:"number"}
MAX_SEQ_LENGTH = 2048  # @param {type:"integer"}
MAX_STEPS = 30  # @param {type:"integer"}
SEED = 3407  # @param {type:"integer"}
RUN_TRAINING = False  # @param {type:"boolean"}
USE_BROWSER_UPLOAD = False  # @param {type:"boolean"}
EXPORT_MERGED_16BIT = False  # @param {type:"boolean"}
EXPORT_GGUF_Q4_K_M = False  # @param {type:"boolean"}
DOWNLOAD_ARTIFACTS = False  # @param {type:"boolean"}

if DATA_SOURCE not in {"huggingface_replay_packets", "local_records", "schema_example"}:
    raise ValueError("Unsupported DATA_SOURCE")
if min(MAX_REPLAY_GAMES, MAX_PACKETS_PER_GAME, PACKET_SAMPLE_STRIDE, WINDOW_SIZE, WINDOW_STRIDE) < 1:
    raise ValueError("Replay limits and window settings must be positive")
if not 0 <= EVAL_MATCH_FRACTION < 1:
    raise ValueError("EVAL_MATCH_FRACTION must be in [0, 1)")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts: {OUTPUT_DIR.resolve()}")

Artifacts: /content/playable-replays-output


## 2. Download, compact, and validate decoded packets

The pinned Hugging Face batch is a gzip-compressed JSONL file: each line is one game containing an ordered `events` array, and each event has one packet-type key. The adapter reads at most `MAX_REPLAY_GAMES`, bounds packet depth/width, removes likely account-identifier fields, and converts packet `i` into a factual target for the next sampled packet. It does not reconstruct professional intent or create coaching labels.

For `local_records`, input may be JSONL, a JSON array, or `{"records": [...]}`. Each record must contain:

```json
{
  "matchId": "licensed-or-authored-source-id",
  "tick": 123,
  "state": {"units": [], "objectives": {}},
  "label": {
    "analysis": "source-backed target text",
    "task": "state_analysis",
    "sourceType": "caster_transcript_alignment",
    "evidenceId": "caption-123",
    "rejectedAnalysis": "optional lower-quality answer for preference export"
  },
  "metadata": {}
}
```

Allowed `sourceType` values are `authored_fixture`, `licensed_replay_export`, `caster_transcript_alignment`, and the non-training `schema_example`. Authored fixtures must disclose an approximation in `metadata.coordinateMethod`. Dataset coordinates remain explicitly marked as dataset-native `x`/`z`, not normalized simulator coordinates.

In [3]:
import gzip
import hashlib
import json
import math
from collections import Counter
from copy import deepcopy
from typing import Any
from urllib.parse import quote
from urllib.request import Request, urlopen

ALLOWED_SOURCE_TYPES = {
    "authored_fixture",
    "licensed_replay_export",
    "caster_transcript_alignment",
    "schema_example",
}
TRAINABLE_SOURCE_TYPES = ALLOWED_SOURCE_TYPES - {"schema_example"}
ALLOWED_TASKS = {"state_analysis", "next_sampled_packet_prediction"}
RAW_REPLAY_SUFFIXES = {".rofl", ".dem", ".mp4", ".mkv", ".mov"}
MAX_DATASET_DOWNLOAD_BYTES = 120_000_000
MAX_PACKET_DEPTH = 4
MAX_PACKET_MAPPING_ITEMS = 24
MAX_PACKET_LIST_ITEMS = 16
MAX_PACKET_STRING_LENGTH = 160
SENSITIVE_KEY_FRAGMENTS = ("summoner", "puuid", "account", "player_name", "riot_id")

SCHEMA_EXAMPLE = [
    {
        "matchId": "schema-smoke-test",
        "tick": tick,
        "state": {
            "mapBounds": {"minX": 0, "maxX": 100, "minY": 0, "maxY": 100},
            "units": [
                {"id": "blue-controlled", "team": "blue", "position": {"x": 40 + tick, "y": 50}, "hp": 80, "gold": 5000 + 25 * tick, "items": ["example-item"], "cooldowns": {"ultimate": max(0, 2 - tick)}},
                {"id": "red-opponent", "team": "red", "position": {"x": 60 - tick, "y": 50}, "hp": 75, "gold": 4900, "items": [], "cooldowns": {"ultimate": 0}},
            ],
            "objectives": {"exampleObjectiveSeconds": 30 - tick},
            "vision": {"blueVisibleRedIds": ["red-opponent"]},
        },
        "label": {
            "analysis": f"Schema smoke test at tick {tick}; do not use this text for training.",
            "task": "state_analysis",
            "sourceType": "schema_example",
            "evidenceId": f"schema-{tick}",
        },
        "metadata": {"coordinateMethod": "synthetic schema example; not match data", "task": "state_analysis"},
    }
    for tick in range(3)
]

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download_replay_batch() -> Path:
    cache_path = OUTPUT_DIR / "hf-cache" / HF_DATASET_REVISION / HF_REPLAY_FILE
    if cache_path.exists():
        actual_hash = sha256_file(cache_path)
        if actual_hash != HF_REPLAY_SHA256:
            raise ValueError(f"Cached dataset checksum mismatch at {cache_path}; remove it manually before retrying")
        print(f"Using verified cached replay batch: {cache_path}")
        return cache_path
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    encoded_file = quote(HF_REPLAY_FILE, safe="/")
    url = f"https://huggingface.co/datasets/{HF_DATASET_REPO}/resolve/{HF_DATASET_REVISION}/{encoded_file}?download=true"
    request = Request(url, headers={"User-Agent": "playable-replays-offline-dataset/1.0"})
    partial_path = cache_path.with_name(cache_path.name + ".part")
    total_bytes = 0
    print(f"Downloading pinned dataset batch: {HF_REPLAY_FILE}")
    with urlopen(request, timeout=60) as response, partial_path.open("wb") as output:
        while chunk := response.read(1024 * 1024):
            total_bytes += len(chunk)
            if total_bytes > MAX_DATASET_DOWNLOAD_BYTES:
                raise ValueError("Dataset batch exceeded the configured download cap")
            output.write(chunk)
    actual_hash = sha256_file(partial_path)
    if actual_hash != HF_REPLAY_SHA256:
        raise ValueError(f"Downloaded dataset checksum mismatch: {actual_hash}")
    partial_path.replace(cache_path)
    print(f"Verified {total_bytes:,} bytes with SHA-256 {actual_hash}")
    return cache_path

def compact_packet_value(value: Any, depth: int = 0) -> Any:
    if value is None or isinstance(value, (bool, int)):
        return value
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, str):
        return value[:MAX_PACKET_STRING_LENGTH]
    if depth >= MAX_PACKET_DEPTH:
        return "<depth-truncated>"
    if isinstance(value, dict):
        compacted: dict[str, Any] = {}
        for raw_key in sorted(value, key=lambda item: str(item)):
            key = str(raw_key)
            lowered = key.lower()
            if lowered == "name" or any(fragment in lowered for fragment in SENSITIVE_KEY_FRAGMENTS):
                continue
            compacted[key[:80]] = compact_packet_value(value[raw_key], depth + 1)
            if len(compacted) >= MAX_PACKET_MAPPING_ITEMS:
                break
        if len(value) > len(compacted):
            compacted["_truncatedOrRedactedItems"] = len(value) - len(compacted)
        return compacted
    if isinstance(value, list):
        compacted = [compact_packet_value(item, depth + 1) for item in value[:MAX_PACKET_LIST_ITEMS]]
        if len(value) > MAX_PACKET_LIST_ITEMS:
            compacted.append({"_truncatedItems": len(value) - MAX_PACKET_LIST_ITEMS})
        return compacted
    return repr(value)[:MAX_PACKET_STRING_LENGTH]

def compact_event(event: Any, game_index: int, packet_index: int) -> dict[str, Any]:
    if not isinstance(event, dict) or len(event) != 1:
        raise ValueError(f"game {game_index} packet {packet_index}: expected exactly one packet-type key")
    packet_type, payload = next(iter(event.items()))
    if not isinstance(packet_type, str) or not packet_type:
        raise ValueError(f"game {game_index} packet {packet_index}: invalid packet type")
    return {"packetType": packet_type, "payload": compact_packet_value(payload)}

def records_from_replay_batch(path: Path) -> list[dict[str, Any]]:
    dataset_url = f"https://huggingface.co/datasets/{HF_DATASET_REPO}"
    converted: list[dict[str, Any]] = []
    loaded_games = 0
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        for source_line, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            if loaded_games >= MAX_REPLAY_GAMES:
                break
            if len(line) > 64_000_000:
                raise ValueError(f"dataset line {source_line} exceeds the 64 MB safety cap")
            game = json.loads(line)
            events = game.get("events") if isinstance(game, dict) else None
            if not isinstance(events, list):
                raise ValueError(f"dataset line {source_line}: events must be an array")
            game_index = loaded_games
            loaded_games += 1
            packet_limit = min(len(events), MAX_PACKETS_PER_GAME)
            sampled = [
                (packet_index, compact_event(events[packet_index], game_index, packet_index))
                for packet_index in range(0, packet_limit, PACKET_SAMPLE_STRIDE)
            ]
            stable_source = f"{HF_DATASET_REPO}@{HF_DATASET_REVISION}:{HF_REPLAY_FILE}:{game_index}"
            match_id = "hf-replay-" + hashlib.sha256(stable_source.encode("utf-8")).hexdigest()[:16]
            for (packet_index, packet), (next_index, next_packet) in zip(sampled, sampled[1:]):
                evidence_id = f"{dataset_url}/blob/{HF_DATASET_REVISION}/{HF_REPLAY_FILE}#game-{game_index}-packet-{next_index}"
                converted.append({
                    "matchId": match_id,
                    "tick": packet_index,
                    "state": {
                        "task": "next_sampled_packet_prediction",
                        "packetIndex": packet_index,
                        "sampleStride": PACKET_SAMPLE_STRIDE,
                        "currentPacket": packet,
                    },
                    "label": {
                        "analysis": json.dumps({"nextSampledPacket": next_packet}, sort_keys=True, separators=(",", ":")),
                        "task": "next_sampled_packet_prediction",
                        "sourceType": "licensed_replay_export",
                        "evidenceId": evidence_id,
                    },
                    "metadata": {
                        "task": "next_sampled_packet_prediction",
                        "datasetRepo": HF_DATASET_REPO,
                        "datasetRevision": HF_DATASET_REVISION,
                        "datasetFile": HF_REPLAY_FILE,
                        "datasetFileSha256": HF_REPLAY_SHA256,
                        "datasetLicense": "Apache-2.0 (publisher-declared)",
                        "datasetUrl": dataset_url,
                        "sourceLine": source_line,
                        "targetPacketIndex": next_index,
                        "coordinateMethod": "dataset-native x/z packet coordinates; not normalized simulator coordinates",
                        "uncertainty": "decoded packet sequence target; no coaching or player-intent label",
                    },
                })
    if not converted:
        raise ValueError("The selected replay batch and limits produced no packet transitions")
    print(f"Converted {loaded_games} games into {len(converted)} bounded packet transitions")
    return converted

def read_records(path: Path) -> list[dict[str, Any]]:
    if path.suffix.lower() in RAW_REPLAY_SUFFIXES:
        raise ValueError("Raw replay/video parsing is intentionally out of scope; provide a licensed structured JSON/JSONL export.")
    if path.suffix.lower() == ".jsonl":
        return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    payload = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and isinstance(payload.get("records"), list):
        return payload["records"]
    raise ValueError("JSON input must be an array or an object containing a records array")

def validate_record(record: dict[str, Any], index: int) -> dict[str, Any]:
    if not isinstance(record, dict):
        raise ValueError(f"record {index}: expected an object")
    match_id = record.get("matchId")
    tick = record.get("tick")
    state = record.get("state")
    label = record.get("label")
    metadata = record.get("metadata", {})
    if not isinstance(match_id, str) or not match_id.strip():
        raise ValueError(f"record {index}: matchId must be a non-empty string")
    if not isinstance(tick, int) or isinstance(tick, bool) or tick < 0:
        raise ValueError(f"record {index}: tick must be a non-negative integer")
    if not isinstance(state, dict) or not state:
        raise ValueError(f"record {index}: state must be a non-empty object")
    if not isinstance(label, dict):
        raise ValueError(f"record {index}: label must be an object")
    if not isinstance(label.get("analysis"), str) or not label["analysis"].strip():
        raise ValueError(f"record {index}: label.analysis must be non-empty")
    task = label.get("task", "state_analysis")
    if task not in ALLOWED_TASKS:
        raise ValueError(f"record {index}: unsupported label.task {task!r}")
    source_type = label.get("sourceType")
    if source_type not in ALLOWED_SOURCE_TYPES:
        raise ValueError(f"record {index}: unsupported label.sourceType {source_type!r}")
    if not isinstance(label.get("evidenceId"), str) or not label["evidenceId"].strip():
        raise ValueError(f"record {index}: label.evidenceId must be non-empty")
    if not isinstance(metadata, dict):
        raise ValueError(f"record {index}: metadata must be an object")
    if source_type == "authored_fixture" and "approx" not in str(metadata.get("coordinateMethod", "")).lower():
        raise ValueError(f"record {index}: authored coordinates must explicitly disclose an approximation")
    if task == "next_sampled_packet_prediction":
        if source_type != "licensed_replay_export":
            raise ValueError(f"record {index}: packet prediction requires licensed_replay_export provenance")
        required_provenance = {"datasetRepo", "datasetRevision", "datasetFileSha256", "datasetLicense"}
        if not required_provenance <= set(metadata):
            raise ValueError(f"record {index}: packet prediction is missing dataset provenance")
    rejected = label.get("rejectedAnalysis")
    if rejected is not None and (not isinstance(rejected, str) or not rejected.strip()):
        raise ValueError(f"record {index}: rejectedAnalysis must be a non-empty string when supplied")
    return deepcopy(record)

if USE_BROWSER_UPLOAD:
    if not IN_COLAB:
        raise RuntimeError("Browser upload is available only inside a Colab runtime")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one JSON or JSONL dataset")
    DATA_PATH = next(iter(uploaded))
    DATA_SOURCE = "local_records"

if DATA_SOURCE == "huggingface_replay_packets":
    raw_records = records_from_replay_batch(download_replay_batch())
elif DATA_SOURCE == "local_records":
    if not DATA_PATH:
        raise ValueError("DATA_PATH is required when DATA_SOURCE is local_records")
    raw_records = read_records(Path(DATA_PATH))
else:
    raw_records = SCHEMA_EXAMPLE
records = [validate_record(record, index) for index, record in enumerate(raw_records)]
records.sort(key=lambda item: (item["matchId"], item["tick"]))
keys = [(item["matchId"], item["tick"]) for item in records]
if len(keys) != len(set(keys)):
    raise ValueError("Duplicate (matchId, tick) records are not allowed")
source_counts = Counter(item["label"]["sourceType"] for item in records)
print(f"Validated {len(records)} ticks across {len(set(item['matchId'] for item in records))} match/source groups")
print(dict(source_counts))
if DATA_SOURCE == "schema_example":
    print("Using the non-training schema smoke test. Select huggingface_replay_packets for the pinned public dataset.")

Verified 86,557,239 bytes with SHA-256 a0fdcb5fc12182fbdb8be0ce8c56c4490e5bd03abc7ccfeb82fe5c1346e3edbe


ValueError: dataset line 2 exceeds the 64 MB safety cap

## 3. Temporal windowing, alignment, and JSONL export

Windows never cross a game-derived `matchId` boundary. The split is grouped by game to reduce temporal leakage. Replay-packet examples predict the next sampled decoded packet as structured JSON; they are not coaching recommendations. Custom labelled examples retain `sourceType` and evidence IDs, and optional `rejectedAnalysis` values are exported separately for later preference work.

In [ ]:
import random
from collections import defaultdict

ANALYSIS_SYSTEM_PROMPT = (
    "Analyze a temporal window of structured game state. Return concise JSON with an analysis string, "
    "task, evidenceIds array, sourceType, and uncertainty string. Separate direct state observations from inference. "
    "Authored coordinates are normalized approximations unless metadata explicitly identifies a licensed replay export. "
    "Do not claim calibrated win probability, optimal real-world play, or professional-player intent."
)
PACKET_SYSTEM_PROMPT = (
    "Given a temporal window of compacted, sampled decoded replay packets, predict only the next sampled packet. "
    "Return JSON with task, prediction, evidenceIds, sourceType, and uncertainty. Preserve packet field names and "
    "dataset-native x/z coordinates exactly as provided. Do not add coaching advice, player intent, win probability, "
    "or claims about events beyond the packet target."
)

def build_windows(items: list[dict[str, Any]]) -> list[dict[str, Any]]:
    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for item in items:
        grouped[item["matchId"]].append(item)
    examples: list[dict[str, Any]] = []
    for match_id in sorted(grouped):
        ticks = sorted(grouped[match_id], key=lambda item: item["tick"])
        for end_index in range(WINDOW_SIZE - 1, len(ticks), WINDOW_STRIDE):
            window = ticks[end_index - WINDOW_SIZE + 1 : end_index + 1]
            target = window[-1]
            task = target["label"].get("task", "state_analysis")
            source_type = target["label"]["sourceType"]
            evidence_ids = [item["label"]["evidenceId"] for item in window]
            if any(item["label"].get("task", "state_analysis") != task for item in window):
                raise ValueError(f"mixed tasks within match/source group {match_id}")
            state_window = {
                "schemaVersion": "1.0",
                "stateScope": "offline_decoded_replay_packets" if task == "next_sampled_packet_prediction" else "offline_authored_or_licensed_export",
                "task": task,
                "matchId": match_id,
                "snapshots": [
                    {"tick": item["tick"], "state": item["state"], "metadata": item.get("metadata", {})}
                    for item in window
                ],
            }
            assistant: dict[str, Any] = {
                "task": task,
                "evidenceIds": evidence_ids,
                "sourceType": source_type,
                "uncertainty": target.get("metadata", {}).get("uncertainty", "not supplied"),
            }
            if task == "next_sampled_packet_prediction":
                assistant["prediction"] = json.loads(target["label"]["analysis"])
                system_prompt = PACKET_SYSTEM_PROMPT
            else:
                assistant["analysis"] = target["label"]["analysis"]
                system_prompt = ANALYSIS_SYSTEM_PROMPT
            examples.append({
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": json.dumps(state_window, sort_keys=True, separators=(",", ":"))},
                    {"role": "assistant", "content": json.dumps(assistant, sort_keys=True)},
                ],
                "metadata": {
                    "matchId": match_id,
                    "startTick": window[0]["tick"],
                    "endTick": window[-1]["tick"],
                    "task": task,
                    "sourceType": source_type,
                    "evidenceIds": evidence_ids,
                    "datasetProvenance": {
                        key: target.get("metadata", {})[key]
                        for key in ("datasetRepo", "datasetRevision", "datasetFile", "datasetFileSha256", "datasetLicense", "datasetUrl")
                        if key in target.get("metadata", {})
                    },
                },
                "preference": target["label"].get("rejectedAnalysis"),
            })
    return examples

def grouped_split(items: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    match_ids = sorted({item["metadata"]["matchId"] for item in items})
    if len(match_ids) < 2 or EVAL_MATCH_FRACTION == 0:
        return items, []
    generator = random.Random(SEED)
    generator.shuffle(match_ids)
    eval_count = max(1, round(len(match_ids) * EVAL_MATCH_FRACTION))
    eval_ids = set(match_ids[:eval_count])
    return (
        [item for item in items if item["metadata"]["matchId"] not in eval_ids],
        [item for item in items if item["metadata"]["matchId"] in eval_ids],
    )

def write_jsonl(path: Path, items: list[dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for item in items:
            handle.write(json.dumps(item, ensure_ascii=False, separators=(",", ":")) + "\n")

all_windows = build_windows(records)
trainable_windows = [item for item in all_windows if item["metadata"]["sourceType"] in TRAINABLE_SOURCE_TYPES]
train_examples, eval_examples = grouped_split(trainable_windows)
preference_examples = [
    {
        "prompt": item["messages"][:2],
        "chosen": item["messages"][2]["content"],
        "rejected": json.dumps({"analysis": item["preference"]}, sort_keys=True),
        "metadata": item["metadata"],
    }
    for item in trainable_windows
    if item.get("preference")
]

write_jsonl(OUTPUT_DIR / "windows.preview.jsonl", all_windows)
write_jsonl(OUTPUT_DIR / "train.jsonl", train_examples)
write_jsonl(OUTPUT_DIR / "eval.jsonl", eval_examples)
write_jsonl(OUTPUT_DIR / "preferences.jsonl", preference_examples)
dataset_report = {
    "ticks": len(records),
    "windows": len(all_windows),
    "trainableWindows": len(trainable_windows),
    "train": len(train_examples),
    "eval": len(eval_examples),
    "preferences": len(preference_examples),
}
print(json.dumps(dataset_report, indent=2))
if RUN_TRAINING and len(train_examples) < 2:
    raise ValueError("Training requires at least two non-example temporal windows")

In [ ]:
# Inspect one aligned pair before downloading a model.
if not all_windows:
    raise ValueError("No temporal windows were produced; lower WINDOW_SIZE or provide more ticks per matchId")
preview = all_windows[0]
print("INPUT WINDOW:")
print(json.dumps(json.loads(preview["messages"][1]["content"]), indent=2)[:4000])
print("\nTARGET:")
print(json.dumps(json.loads(preview["messages"][2]["content"]), indent=2))

## 4. Optional QLoRA fine-tuning with Unsloth

The install follows Unsloth's current `pip install unsloth` guidance and uses the instruction-tuned 4-bit model named above. Import Unsloth before TRL so its patches apply. Training is deliberately gated by `RUN_TRAINING`; schema examples cannot pass the training guard. Model downloads and license acceptance remain the operator's responsibility.

In [ ]:
import subprocess

if RUN_TRAINING:
    if not cuda_available:
        raise RuntimeError("QLoRA training requires an assigned CUDA GPU runtime")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "unsloth", "datasets", "trl"])
    print("Training dependencies installed. If imports fail due to a runtime package replacement, restart the runtime once.")
else:
    print("Dependency installation skipped (RUN_TRAINING=False).")

In [ ]:
model = tokenizer = trainer = None
train_dataset = eval_dataset = None

if RUN_TRAINING:
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from datasets import Dataset
    from transformers import TrainingArguments
    from trl import SFTTrainer

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=False,
    )

    def to_text(batch: dict[str, list[Any]]) -> dict[str, list[str]]:
        return {
            "text": [
                tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
                for messages in batch["messages"]
            ]
        }

    train_dataset = Dataset.from_list(train_examples).map(to_text, batched=True)
    eval_dataset = Dataset.from_list(eval_examples).map(to_text, batched=True) if eval_examples else None
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=1,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=min(5, max(1, MAX_STEPS // 10)),
            max_steps=MAX_STEPS,
            learning_rate=2e-4,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=SEED,
            output_dir=str(OUTPUT_DIR / "checkpoints"),
            report_to="none",
        ),
    )
    print(f"Prepared {len(train_dataset)} training examples and {len(eval_dataset) if eval_dataset is not None else 0} eval examples")
else:
    print("Model setup skipped (RUN_TRAINING=False).")

In [ ]:
training_metrics = None
if RUN_TRAINING:
    training_result = trainer.train()
    training_metrics = training_result.metrics
    print(json.dumps(training_metrics, indent=2, default=str))
else:
    print("Training skipped. Validate the exported JSONL, attach a GPU, then set RUN_TRAINING=True.")

## 5. Evaluation and provenance check

A single generation is a smoke test, not evidence of model quality. Keep entire games out of training. For decoded packets, report exact packet-type accuracy plus field-level precision/recall against the held-out next-packet target. For analysis-labelled custom data, score factual grounding, evidence-ID retention, JSON validity, temporal ordering, and unsupported intent/optimality claims with expert review.

In [ ]:
generation_report = None
if RUN_TRAINING:
    FastLanguageModel.for_inference(model)
    candidate = (eval_examples or train_examples)[0]
    prompt_messages = candidate["messages"][:2]
    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    ).to(model.device)
    outputs = model.generate(input_ids=inputs, max_new_tokens=256, do_sample=False, use_cache=True)
    generated_text = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    try:
        parsed = json.loads(generated_text)
        task = candidate["metadata"]["task"]
        required = {"task", "evidenceIds", "sourceType", "uncertainty"}
        required.add("prediction" if task == "next_sampled_packet_prediction" else "analysis")
        generation_report = {"validJson": True, "hasRequiredKeys": required <= set(parsed), "output": parsed}
    except json.JSONDecodeError:
        generation_report = {"validJson": False, "output": generated_text}
    print(json.dumps(generation_report, indent=2, ensure_ascii=False))
else:
    print("Generation skipped (RUN_TRAINING=False).")

## 6. Export

The default export is the small LoRA adapter plus a provenance manifest. A merged 16-bit model needs substantially more disk/RAM. For local Ollama-style use, `q4_k_m` GGUF is the safer optional quantized path; blindly merging directly to 4-bit can degrade quality, so it is not the default. Never put Hugging Face or other API tokens in this notebook.

In [ ]:
import shutil
from datetime import datetime, timezone

if RUN_TRAINING:
    adapter_dir = OUTPUT_DIR / "esports-tracker-lora"
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    unique_dataset_provenance = {
        json.dumps(item["metadata"]["datasetProvenance"], sort_keys=True): item["metadata"]["datasetProvenance"]
        for item in train_examples
        if item["metadata"].get("datasetProvenance")
    }
    manifest = {
        "createdAt": datetime.now(timezone.utc).isoformat(),
        "baseModel": BASE_MODEL,
        "method": "QLoRA",
        "seed": SEED,
        "windowSize": WINDOW_SIZE,
        "trainExamples": len(train_examples),
        "evalExamples": len(eval_examples),
        "sourceTypes": sorted({item["metadata"]["sourceType"] for item in train_examples}),
        "tasks": sorted({item["metadata"]["task"] for item in train_examples}),
        "datasetProvenance": list(unique_dataset_provenance.values()),
        "trainingMetrics": training_metrics,
        "disclosure": "Decoded-packet outputs are sequence predictions, not coaching facts or runtime telemetry. Packet-native x/z coordinates are not normalized simulator coordinates.",
    }
    (adapter_dir / "training-manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

    if EXPORT_MERGED_16BIT:
        model.save_pretrained_merged(str(OUTPUT_DIR / "esports-tracker-merged-16bit"), tokenizer, save_method="merged_16bit")
    if EXPORT_GGUF_Q4_K_M:
        model.save_pretrained_gguf(str(OUTPUT_DIR / "esports-tracker-gguf"), tokenizer, quantization_method="q4_k_m")

    archive = shutil.make_archive(str(OUTPUT_DIR / "esports-tracker-lora"), "zip", root_dir=adapter_dir)
    print(f"Saved adapter archive: {archive}")
    if DOWNLOAD_ARTIFACTS:
        if not IN_COLAB:
            raise RuntimeError("Automatic browser download is available only in Colab")
        from google.colab import files
        files.download(archive)
else:
    print(f"Dataset artifacts are available in {OUTPUT_DIR.resolve()}; model export skipped.")

## Run checklist

1. In VS Code, open this notebook and choose **Select Kernel → Colab → Auto Connect**. Sign in only through the Google extension UI.
2. Run the connectivity cell and confirm `inColab: true`, a CUDA accelerator, and reachable dataset/model/package endpoints.
3. Keep `DATA_SOURCE=huggingface_replay_packets` to download the pinned `12_22/batch_001.jsonl.gz`, verify its published SHA-256, and stream the configured game limit. The file is about 86.6 MB; the full repository is much larger and is intentionally not downloaded.
4. Inspect the preview: inputs must contain only compacted packet windows and targets must be factual next-sampled-packet JSON. For custom data, switch to `local_records` and use the Colab activity bar's **Upload to Colab** action or `USE_BROWSER_UPLOAD=True`.
5. Use multiple games for leakage-resistant evaluation; substantially more than two windows is required for a meaningful model.
6. Set `RUN_TRAINING=True`, rerun from configuration onward, evaluate, and export the adapter.

### Current references

- [Google Colab FAQ](https://research.google.com/colaboratory/faq.html) — hosted notebooks, GitHub loading, dynamic accelerator availability, and resource limits.
- [Google Colab VS Code extension](https://github.com/googlecolab/colab-vscode) — kernel connection workflow.
- [Decoded replay packet dataset](https://huggingface.co/datasets/maknee/league-of-legends-decoded-replay-packets) — packet schema, declared Apache-2.0 license, citation, and recommended batch download flow. Dataset citation: `maknee (2025), League of Legends Decoded Replay Packets Dataset`.
- [Pinned default batch](https://huggingface.co/datasets/maknee/league-of-legends-decoded-replay-packets/blob/04f9c7350e9ffcc689b731875ad9baf3ff6eaa6d/12_22/batch_001.jsonl.gz) — 86.6 MB Xet object, SHA-256 `a0fdcb5fc12182fbdb8be0ce8c56c4490e5bd03abc7ccfeb82fe5c1346e3edbe`.
- [Unsloth installation](https://docs.unsloth.ai/get-started/installing-%2B-updating/pip-install) — current notebook installation guidance.
- [Unsloth Llama 3.2 3B Instruct 4-bit model](https://huggingface.co/unsloth/Llama-3.2-3B-Instruct-bnb-4bit) — model card, license, and Colab entry point.

Free Colab does not guarantee a particular GPU, VRAM amount, run duration, or availability. The timings and fixed T4 claims in the pasted guide should be treated as rough historical examples, not promises.